#CSI INACA Data Preparation

Kode berikut melakukan data preparation untuk analisis bisnis berbasis Customer Satisfaction Index (CSI) INACA.

Tahapannya meliputi:

Load dan autentikasi data dari Google Sheets.

Definisi KPI dan pengelompokan kolom survei.

Pembersihan data: konversi tipe data, isi nilai kosong, dan standar kolom kategorikal.

Perhitungan CSI Score dan KPI per bandara.

Transformasi data ke long format untuk visualisasi.

Upload hasil ke Google Sheets untuk dashboard analisis selanjutnya.

Hasil dari tahap ini adalah dataset bersih dan terstruktur siap digunakan untuk visualisasi KPI dan CSI.

In [ ]:
#Install and import libraries
!pip install --upgrade gspread pandas gspread_dataframe

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from gspread_dataframe import get_as_dataframe, set_with_dataframe
import pandas as pd

In [ ]:
# Google Sheets Authentication
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# Load worksheets as DataFrames
sheet = gc.open_by_key('YOUR_SHEET_KEY_HERE')  # replace with your actual sheet key
worksheet1 = sheet.worksheet('KAMUS')
worksheet2 = sheet.worksheet('PAX (ALL)')

df_kamus = get_as_dataframe(worksheet1)
df_pax = get_as_dataframe(worksheet2)

In [ ]:
# KPI Groups Definition
kpi_groups = {
    "Hospitality": ['E','F','G','H','K','M','N','P','R','U','KK'],
    "Health & Safety Protocol": ['A','B','C','D','I','J'],
    "Service": ['L','O','Q','JJ'],
    "Facilities": ['S','T','V','W','X','Y','Z','AA','BB','CC','DD','EE','FF','GG'],
    "Terminal Environment": ['HH','II'],
    "Overall Satisfaction": ['UMUM']
}

In [ ]:
# Data Cleaning
# Convert date and time
df_pax['DATE'] = pd.to_datetime(df_pax['DATE'], errors='coerce')
df_pax['JAM'] = pd.to_numeric(df_pax['JAM'], errors='coerce')

# Convert coordinates
df_pax['LAT_LNG'] = pd.to_numeric(df_pax['LAT_LNG'], errors='coerce')
df_pax['LAT'] = pd.to_numeric(df_pax['LAT'], errors='coerce')
df_pax['LNG'] = pd.to_numeric(df_pax['LNG'], errors='coerce')

# Convert categorical columns to string
categorical_cols = ['AIRLINE_ORIGIN', '1 (TUJUAN)', '3 (TRANSPORTASI)', '5 (AJTIVITAS)',
                    '6 (BELANJA)', '7 (TIDAK BELANJA)', 'PEKERJAAN', 'GATE', 'SURVEYOR']

for col in categorical_cols:
    df_pax[col] = df_pax[col].astype(str)

# Fill missing numeric values and set YEAR as Int64
df_pax = df_pax.fillna({"JAM":0, "YEAR":0, "LAT":0.0, "LNG":0.0})
df_pax["YEAR"] = df_pax["YEAR"].astype("Int64")

In [ ]:
# CSI Score Calculation
cols_csi = ['A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P','Q','R','S','T',
            'U','V','W','X','Y','Z','AA','BB','CC','DD','EE','FF','GG','HH','II','JJ','KK','UMUM']

df_pax[cols_csi] = df_pax[cols_csi].apply(pd.to_numeric, errors='coerce')
df_pax['CSI SCORE'] = df_pax[cols_csi].mean(axis=1, skipna=True)

In [ ]:
# Define clean copy and airport list
df_pax_clean = df_pax.copy()
bandara_list = df_pax_clean['BANDAR'].dropna().unique()

In [ ]:
# Calculate KPI per airport
results = []

for bandara in bandara_list:
    df_bandara = df_pax_clean[df_pax_clean['BANDAR']==bandara]
    row = {'Bandara': bandara}

    for kpi, col_list in kpi_groups.items():
        valid_cols = [col for col in col_list if col in df_bandara.columns]
        if valid_cols:
            sub_df = df_bandara[valid_cols].apply(pd.to_numeric, errors='coerce')
            row[kpi] = round(sub_df.mean().mean(),2)
            for col in valid_cols:
                row[f"{kpi} - {col}"] = round(sub_df[col].mean(),2)
    results.append(row)

df_result = pd.DataFrame(results)

In [ ]:
# Group KPI breakdown and calculate KPI average
all_columns = df_result.columns.tolist()
kpi_core_columns = {kpi:[col for col in all_columns if col.startswith(f"{kpi} - ")] for kpi in kpi_groups.keys()}

for kpi, col_list in kpi_core_columns.items():
    df_result[kpi] = df_result[col_list].mean(axis=1).round(2)

In [ ]:
# Convert df_result to long format
df_long = []

for i,row in df_result.iterrows():
    bandara = row['Bandara']
    for kpi, elemen_cols in kpi_core_columns.items():
        skor_kpi = row[kpi]
        for elemen in elemen_cols:
            skor_elemen = row[elemen]
            df_long.append({
                'Bandara': bandara,
                'KPI': kpi,
                'Skor KPI': skor_kpi,
                'Elemen': elemen.split(" - ",1)[-1],
                'Skor Elemen': skor_elemen
            })

df_kpi_long = pd.DataFrame(df_long)

In [ ]:
# Upload df_result and df_kpi_long to Google Sheets
try:
    worksheet_result = sheet.worksheet("KPI CSI Summary")
    worksheet_result.clear()
except gspread.exceptions.WorksheetNotFound:
    worksheet_result = sheet.add_worksheet(title="KPI CSI Summary", rows=1000, cols=50)

set_with_dataframe(worksheet_result, df_result)

try:
    worksheet_long = sheet.worksheet("KPI CSI Long")
    worksheet_long.clear()
except gspread.exceptions.WorksheetNotFound:
    worksheet_long = sheet.add_worksheet(title="KPI CSI Long", rows=1000, cols=50)

set_with_dataframe(worksheet_long, df_kpi_long)

In [ ]:
# Write CSI SCORE to worksheet starting from column 'CM2'
csi_values = df_pax['CSI SCORE'].apply(lambda x:[x]).tolist()
worksheet2.update('CM2', csi_values)